# Insurance Underwriting Governance Review

Decision support for rate and underwriting governance using a freMTPL-style motor portfolio. Target: `material_loss_flag`. Outcome fields such as claim count, claim amount, and annualized loss cost are excluded from predictors to prevent leakage.

## Governance scope

This notebook is intended for governance, audit, and business review. It uses HUG-IML as the modeling API. 

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, precision_recall_curve, confusion_matrix
)

warnings.filterwarnings("ignore")

try:
    from hugiml import HUGIMLClassifierNative
except ImportError as exc:
    raise ImportError(
        "This notebook requires hugiml-core. Install it with: pip install hugiml-core"
    ) from exc


## Load data and define decision variables

In [ ]:
DATA_PATH = Path("nb09_insurance_underwriting_data.csv")

PREDICTORS = [
    "exposure", "area", "region", "vehicle_power", "vehicle_age",
    "driver_age", "bonus_malus", "density", "vehicle_brand"
]
TARGET = "material_loss_flag"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## Fit HUG-IML model and select governed threshold

In [ ]:
X = df[PREDICTORS]
y = df[TARGET]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.30, random_state=23, stratify=y
)

clf = HUGIMLClassifierNative(
    B=10, L=2, G=0.00005, topK=120,
    adaptive_binning=True, b_candidates=[6, 8, 10, 12], n_jobs=1
)
c = clf.fit(X_train, y_train)
scores = clf.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, scores)
valid = np.where(precision[:-1] >= 0.78)[0]
if len(valid):
    threshold = float(thresholds[valid[np.argmax(recall[:-1][valid])]])
else:
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    threshold = float(thresholds[np.argmax(f1)])

pred = (scores >= threshold).astype(int)

metrics = pd.Series({
    "auc": roc_auc_score(y_test, scores),
    "average_precision": average_precision_score(y_test, scores),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
    "threshold": threshold,
    "pattern_count": len(clf.get_hug_features()),
})
metrics


## Risk-band economic evidence

In [ ]:
holdout = df.loc[idx_test].copy()
holdout["score"] = scores
holdout["prediction"] = pred
holdout["actual"] = y_test.values
holdout["risk_band"] = pd.qcut(
    holdout["score"],
    q=[0, 0.50, 0.75, 0.90, 0.97, 1.0],
    labels=["Core book", "Watch list", "Elevated", "Rate action", "Referral"],
    duplicates="drop",
)

band_summary = holdout.groupby("risk_band", observed=True).agg(
    policies=("policy_id", "count"),
    exposure=("exposure", "sum"),
    claim_rate=("claim_count", lambda s: (s > 0).mean()),
    observed_material_loss=("actual", "mean"),
    avg_score=("score", "mean"),
    loss_cost=("annualized_loss_cost", "mean"),
    mean_claim_amount=("claim_amount", "mean"),
).reset_index()
band_summary["loss_relativity"] = band_summary["loss_cost"] / band_summary["loss_cost"].iloc[0]
band_summary


## Proxy-risk review by age band

In [ ]:
holdout["age_band"] = pd.cut(
    holdout["driver_age"],
    bins=[17, 25, 35, 55, 70, 90],
    labels=["18-24", "25-34", "35-54", "55-69", "70+"]
)

age_summary = holdout.groupby("age_band", observed=True).agg(
    policies=("policy_id", "count"),
    review_rate=("prediction", "mean"),
    observed_loss=("actual", "mean"),
    avg_score=("score", "mean"),
    loss_cost=("annualized_loss_cost", "mean"),
).reset_index()
age_summary


## HUG-IML pattern evidence

In [ ]:
patterns = pd.DataFrame({
    "pattern": clf.get_hug_features(),
    "coefficient": clf.model_.named_steps["clf"].coef_[0],
})
patterns["abs_coefficient"] = patterns["coefficient"].abs()
patterns.sort_values("abs_coefficient", ascending=False).head(20)
